# Unit Test Generator

This is a tool that will utilize open & closed source models to generate unit tests for a piece of code.

In [36]:
# imports

import os
import re
import io
import sys
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

In [37]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

In [38]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"

anthropic_client = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)

ollama_url = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=ollama_url, api_key="ollama")

In [39]:
CODE = """
    def add(a,b):
        return a + b
"""

In [40]:
# This is for when displaying in Gradio

def get_ollama_models():
    try:
        response = requests.get(f"{ollama_url.replace('/v1', '')}/api/tags", timeout=2)
        response.raise_for_status()
        return [model["name"] for model in response.json().get("models", [])]
    except Exception:
        return ["llama3.2:latest", "deepseek-r1:1.5b"]

frontier_models = ["gpt-5", "claude-sonnet-4-5"]
ollama_models = get_ollama_models()
models = frontier_models + ollama_models

clients = {
    "gpt-5": openai,
    "claude-sonnet-4-5": anthropic_client,
    **{model: ollama_client for model in ollama_models},
}

In [41]:
MAX_TOKENS = 1500
OPENAI_RESPONSES_MODEL_PREFIXES = ("gpt-", "o1", "o3", "o4", "o5", "chatgpt-")

def uses_openai_responses_api(model_name):
    normalized_name = model_name.lower()
    return normalized_name.startswith(OPENAI_RESPONSES_MODEL_PREFIXES)

def get_client(model_name):
    normalized_name = model_name.lower()

    if model_name in clients:
        return clients[model_name]
    if uses_openai_responses_api(model_name):
        return openai
    if normalized_name.startswith("claude"):
        return anthropic_client
    if ":" in model_name:
        return ollama_client

    raise ValueError(f"No client configured for model: {model_name}")

def build_prompt(code):
    return f"""
You are a senior software engineer writing production-quality unit tests.

Analyze the code snippet and generate a complete, runnable unit test file for it.
Use the language and test framework that best match the snippet; for Python, prefer pytest unless the snippet clearly uses unittest.

Requirements:
- Cover the normal behavior, edge cases, error paths, and boundary values that are relevant to the snippet.
- Use descriptive test names that explain the behavior being verified.
- Keep tests independent: no shared mutable state, order dependencies, or hidden external requirements.
- Mock external dependencies such as network calls, file I/O, time, randomness, and environment variables when needed.
- Include only the imports, fixtures, helpers, and tests required to run the test file.
- If the snippet is incomplete, make the smallest reasonable assumption and document it in a short code comment.
- Return only the raw test file content.
- Do not include markdown code fences, language labels, headings, notes, or explanatory prose.
- Do not start with phrases like "Here is", "Sure", "Notes", or ```language.

<code_to_test>
{code}
</code_to_test>

"""

def stream_comment(code, model_name):
    prompt = build_prompt(code)
    client = get_client(model_name)

    if uses_openai_responses_api(model_name):
        request = {
            "model": model_name,
            "input": prompt,
            # "max_output_tokens": MAX_TOKENS,
            "stream": True,
        }

        if model_name.startswith("gpt-5"):
            request["reasoning"] = {"effort": "minimal"}

        stream = client.responses.create(**request)
        for event in stream:
            if event.type == "response.output_text.delta":
                yield event.delta
            elif event.type == "error":
                raise RuntimeError(event.message)

    else:
        stream = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            # max_tokens=MAX_TOKENS,
            stream=True,
        )
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                yield chunk.choices[0].delta.content

def strip_code_fences(text):
    cleaned = re.sub(r"^\s*```[a-zA-Z0-9_+.#-]*\s*\n?", "", text)
    cleaned = re.sub(r"\n?\s*```\s*$", "", cleaned)
    return cleaned

def generate_comment(code, model_name):
    return strip_code_fences("".join(stream_comment(code, model_name)))

def generate_comment_stream(code, model_name):
    output = ""
    for chunk in stream_comment(code, model_name):
        output += chunk
        yield strip_code_fences(output)

def compare_comment_stream(code, selected_models):
    selected_models = selected_models or []
    outputs = {model: "" for model in models}

    yield tuple(outputs[model] for model in models)

    for model_name in selected_models:
        if model_name not in outputs:
            continue

        try:
            for output in generate_comment_stream(code, model_name):
                outputs[model_name] = output
                yield tuple(outputs[model] for model in models)
        except Exception as error:
            outputs[model_name] = f"# Error from {model_name}\n{error}"
            yield tuple(outputs[model] for model in models)

def display_streaming_comment(code, model_name):
    output = ""
    display_handle = display(Markdown(output), display_id=True)

    for output in generate_comment_stream(code, model_name):
        display_handle.update(Markdown(output))

    return output

In [ ]:
UNIT_TEST_WRITER_CSS = """
.editor-panel,
.output-panel,
.comparison-panel {
    height: 640px;
    max-height: 640px;
    overflow: auto;
}
.editor-panel textarea,
.editor-panel .cm-editor,
.editor-panel .cm-scroller,
.output-panel textarea,
.output-panel .cm-editor,
.output-panel .cm-scroller,
.comparison-panel textarea,
.comparison-panel .cm-editor,
.comparison-panel .cm-scroller {
    height: 100%;
    max-height: 640px;
    overflow: auto;
}
.controls {
    align-items: center;
}
.generate-btn {
    min-height: 42px;
}
"""

with gr.Blocks(css=UNIT_TEST_WRITER_CSS, theme=gr.themes.Monochrome(), title="Unit Test Writer") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            code_input = gr.Code(
                label="Code snippet",
                value=CODE,
                language="python",
                lines=26,
                elem_classes=["editor-panel"]
            )

        with gr.Column(scale=6):
            test_output = gr.Code(
                label="Generated unit tests",
                value="",
                language="python",
                lines=26,
                interactive=False,
                elem_classes=["output-panel"]
            )

    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(
            choices=models,
            value=models[0],
            label="Model",
            allow_custom_value=True
        )
        generate = gr.Button("Generate unit tests", elem_classes=["generate-btn"])

    generate.click(
        fn=generate_comment_stream,
        inputs=[code_input, model],
        outputs=[test_output]
    )

    with gr.Row(elem_classes=["controls"]):
        comparison_models = gr.CheckboxGroup(
            choices=models,
            value=models,
            label="Compare models"
        )
        compare = gr.Button("Compare selected models", elem_classes=["generate-btn"])

    comparison_outputs = []
    with gr.Tabs():
        for model_name in models:
            with gr.Tab(model_name):
                comparison_outputs.append(
                    gr.Code(
                        label=model_name,
                        value="",
                        language="python",
                        lines=26,
                        interactive=False,
                        elem_classes=["comparison-panel"]
                    )
                )

    compare.click(
        fn=compare_comment_stream,
        inputs=[code_input, comparison_models],
        outputs=comparison_outputs
    )

ui.queue().launch(inbrowser=True)